# 5. Jailbreak Scenario

The `Jailbreak` scenario tests whether a target model is susceptible to prompt injection and jailbreak
attacks. It uses template-based jailbreak techniques sourced from `TextJailBreak.get_jailbreak_templates()`,
applying them through different attack types ranging from simple prompt sending to complex multi-turn
strategies.

## Available Strategies

| Strategy | CLI Value | Tags | Description |
|----------|-----------|------|-------------|
| ALL | `all` | all | Aggregate — runs all strategies |
| SIMPLE | `simple` | simple | Aggregate — runs simple strategies |
| COMPLEX | `complex` | complex | Aggregate — runs complex strategies |
| PromptSending | `prompt_sending` | simple | Single-turn with jailbreak template |
| ManyShot | `many_shot` | complex | Multi-turn ManyShot jailbreak |
| SkeletonKey | `skeleton` | complex | SkeletonKey jailbreak technique |
| RolePlay | `role_play` | complex | Role-play based persuasion |

The scenario also accepts `num_templates` to limit how many jailbreak templates are used per strategy,
`num_attempts` to repeat each template multiple times, and `jailbreak_names` to select specific templates
by name.

## Setup

In [1]:
from pyrit.prompt_target import OpenAIChatTarget
from pyrit.scenario import DatasetConfiguration
from pyrit.scenario.printer.console_printer import ConsoleScenarioResultPrinter
from pyrit.scenario.scenarios.airt import Jailbreak, JailbreakStrategy
from pyrit.setup import IN_MEMORY, initialize_pyrit_async
from pyrit.setup.initializers import LoadDefaultDatasets

await initialize_pyrit_async(memory_db_type=IN_MEMORY, initializers=[LoadDefaultDatasets()])  # type: ignore

objective_target = OpenAIChatTarget()
printer = ConsoleScenarioResultPrinter()

Found default environment files: ['./.pyrit/.env', './.pyrit/.env.local']
Loaded environment file: ./.pyrit/.env
Loaded environment file: ./.pyrit/.env.local


Loading datasets - this can take a few minutes:   0%|                                       | 0/58 [00:00<?, ?dataset/s]

Loading datasets - this can take a few minutes:   2%|▌                              | 1/58 [00:00<00:14,  4.00dataset/s]

Loading datasets - this can take a few minutes:   7%|██▏                            | 4/58 [00:00<00:04, 12.94dataset/s]

Loading datasets - this can take a few minutes:  10%|███▏                           | 6/58 [00:00<00:03, 14.40dataset/s]

...

Loading datasets - this can take a few minutes:  26%|███████▊                      | 15/58 [00:00<00:02, 17.24dataset/s]

Loading datasets - this can take a few minutes:  33%|█████████▊                    | 19/58 [00:01<00:01, 22.41dataset/s]

Loading datasets - this can take a few minutes:  40%|███████████▉                  | 23/58 [00:01<00:01, 25.81dataset/s]

Loading datasets - this can take a few minutes:  45%|█████████████▍                | 26/58 [00:01<00:01, 24.17dataset/s]

Loading datasets - this can take a few minutes: 100%|██████████████████████████████| 58/58 [00:01<00:00, 40.36dataset/s]

## Running via CLI

To run only the fast single-turn strategy:

```bash
pyrit_scan airt.jailbreak \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies prompt_sending \
  --max-dataset-size 2
```

To run all simple strategies:

```bash
pyrit_scan airt.jailbreak \
  --initializers target load_default_datasets \
  --target openai_chat \
  --strategies simple \
  --max-dataset-size 2
```

## Programmatic Usage

Here we run `prompt_sending` with a small dataset and limit to 1 jailbreak template for speed.

In [2]:
dataset_config = DatasetConfiguration(dataset_names=["airt_harms"], max_dataset_size=2)

scenario = Jailbreak(num_templates=1)
await scenario.initialize_async(  # type: ignore
    objective_target=objective_target,
    scenario_strategies=[JailbreakStrategy.PromptSending],
    dataset_config=dataset_config,
)

print(f"Scenario: {scenario.name}")
print(f"Atomic attacks: {scenario.atomic_attack_count}")

Scenario: Jailbreak
Atomic attacks: 1


In [3]:
scenario_result = await scenario.run_async()  # type: ignore

Executing Jailbreak:   0%|          | 0/1 [00:00<?, ?attack/s]

## Interpreting Results

In [4]:
await printer.print_summary_async(scenario_result)  # type: ignore


                                   📊 SCENARIO RESULTS: Jailbreak                                    

▼ Scenario Information
────────────────────────────────────────────────────────────────────────────────────────────────────
  📋 Scenario Details
    • Name: Jailbreak
    • Scenario Version: 1
    • PyRIT Version: 0.12.1.dev0
    • Description:
        Jailbreak scenario implementation for PyRIT. This scenario tests how vulnerable models are to jailbreak attacks
        by applying various single-turn jailbreak templates to a set of test prompts. The responses are scored to
        determine if the jailbreak was successful.

  🎯 Target Information
    • Target Type: OpenAIChatTarget
    • Target Model: gpt-4o
    • Target Endpoint: https://api.openai.com/v1

  📊 Scorer Information
    ▸ Scorer Identifier
      • Scorer Type: TrueFalseInverterScorer
      • scorer_type: true_false
      • score_aggregator: OR_
        └─ Composite of 1 scorer(s):
            • Scorer Type: SelfAskRefus